# Phase 2: Colab runs (setup, eligibility, L4 pipeline checks)

1. **Setup** (cells 1 to 6): clone, install, GPU/bf16 check, pair eligibility.
2. **L4 pipeline checks** (below): does the transfer pipeline actually work on
   Qwen3-1.7B -> 4B in bf16, and does anything visibly change in refusals? This decides
   whether to pay for Colab Pro+ and an A100 for the anchor reproduction against the paper.

**Before running:** Runtime > Change runtime type > **L4** (native bf16; a T4 has none).
Add an `HF_TOKEN` secret (key icon in the left sidebar) so model downloads are not rate limited.
See `docs/gpu-setup.md`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. (Optional) mount Drive for checkpoints, so a disconnect does not lose the
#    HF cache, the fitted mapper, or results. Skip for the eligibility smoke run.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/kv-transfer-safety/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# 3. Clone the repo (public; no token needed).
import os
REPO = 'TELLON2001/kv-transfer-safety'
if not os.path.isdir('kv-transfer-safety'):
    !git clone https://github.com/{REPO}.git
%cd kv-transfer-safety
!git log --oneline -1

The repo is public, so the clone above needs no auth. If you make it private again before the preprint, use a fine-grained PAT via Colab Secrets (see `docs/colab-github-access.md`).

In [ ]:
# 4. Install dependencies. torch is preinstalled on Colab; the rest come from the repo,
#    including kvtransfer pinned to commit 04a8e99 (see requirements.txt).
!pip install -q -r requirements.txt

In [ ]:
# 5. Confirm CUDA + bf16 support (bf16 matters for a faithful reproduction).
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
# 6. Smoke test: run the eligibility check for the pair. This downloads only the
#    model configs + tokenizers (small), not the weights, and confirms the pair is
#    mappable (shared tokenizer, matched KV heads / head_dim).
from src.config import load_config
from src.eligibility import check_pair

cfg = load_config('configs/qwen3-1p7b-to-4b.yaml')
print(cfg['pair_id'], ':', cfg['sharer']['model_id'], '->', cfg['receiver']['model_id'])
report = check_pair(cfg)
print('checks:', report.checks)
print('OK' if report.ok else ('PROBLEMS: ' + '; '.join(report.problems)))

## L4 pipeline checks: Qwen3-1.7B -> 4B in bf16

The pair fits on an L4 at full precision (about 11.5 GB of weights; the ~15 GB of calibration
statistics accumulate on CPU, per the config). Run the steps in order and stop at the first
failure: each step is only meaningful if the ones before it passed.

| Step | What it checks | Time (rough) |
|---|---|---|
| 1 | kvtransfer's own test suite: identity pair reproduces native logits | minutes |
| 2 | calibrate + fit (quick 100-seq pass, then the paper's 500) | ~15 min, then ~1 h |
| 3 | held-out diagnostics: attention cosine, logit KL, top-1 agreement vs the sharer baseline | minutes |
| 4 | capability retention on HellaSwag + ARC-Challenge (limit 250), three ways | 1 to 2 h |
| 5 | smoke test: 20 chat prompts, native vs mapped vs sharer, read by hand | under 1 h |

None of this reproduces the paper (it never evaluated this pair); it checks plausibility.
Any refusal effect seen here is unreportable until the A100 anchor run validates kvtransfer.

In [ ]:
# Hugging Face token from Colab Secrets (never paste a token into a cell).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set')
except Exception as e:
    print('no HF_TOKEN secret, downloads may be rate limited:', e)

In [ ]:
# (Optional) keep results/ on Drive so a disconnect does not lose the fitted mapper.
# Sizes: mapper k=8 is ~2.4 GB (fp32); calibration stats are ~15 GB, so they stay local
# unless you have the Drive space. Run after the clone (cell 3).
PERSIST_RESULTS = False
if PERSIST_RESULTS:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, shutil
    dst = '/content/drive/MyDrive/kv-transfer-safety/results'
    os.makedirs(dst, exist_ok=True)
    if os.path.isdir('results') and not os.path.islink('results'):
        shutil.rmtree('results')
    if not os.path.exists('results'):
        os.symlink(dst, 'results')
    print('results/ ->', os.path.realpath('results'))

### Step 1: kvtransfer's own tests
The pip install has no tests, so clone the same commit. Expect all passed. The key one is `test_identity_transfer_matches_target_generate`.

In [ ]:
!git clone -q https://github.com/Susmith4710/kvtransfer.git /content/kvtransfer 2>/dev/null; cd /content/kvtransfer && git checkout -q 04a8e99d1e1a364982065b23ca10183483924a97 && git log --oneline -1
!pip install -q pytest
!cd /content/kvtransfer && python -m pytest -q 2>&1 | tail -5

### Step 2a: quick calibration (100 seqs)
Shakes out crashes and memory problems cheaply. Writes to separate `-n100` paths, so it never mixes with the full run.

In [ ]:
!python scripts/reproduce.py --config configs/qwen3-1p7b-to-4b.yaml --n-seqs 100 --fit-only

### Step 3a: diagnostics on the quick mapper
Pipeline check only; the numbers improve with the full calibration.

In [ ]:
!python scripts/diagnose.py --config configs/qwen3-1p7b-to-4b.yaml --mapper results/mappers/qwen3-1p7b-to-4b-n100/k8 --n-seqs 16

### Step 2b: full calibration (paper recipe: 500 x 1,024 tokens, stride 4)

In [ ]:
!python scripts/reproduce.py --config configs/qwen3-1p7b-to-4b.yaml --fit-only

### Step 3b: diagnostics on the full mapper
How to read it:
* **top-1 agree**: `mapped-vs-own` must sit clearly above `source-vs-target` (the 1.7B alone).
  If not, the mapper is not carrying the 4B's state; stop and debug.
* **attn cosine**: the paper's best predictor of retention (Sec. 4.5). Note `min-layer` too.
* **logit KL**: lower is better; compare across k if you sweep.

In [ ]:
!python scripts/diagnose.py --config configs/qwen3-1p7b-to-4b.yaml --n-seqs 32

### Step 4: capability retention (limit 250 per task)
No reference number exists for this pair, so the gate prints CHECK; read the retention line.
The paper's Qwen3 pairs kept ~88 to 98% average accuracy. Mapped accuracy near the 1.7B's
(or below) means something is broken; between the two models or near the 4B is plausible.
At n=250 the 95% CI is about +/-6 points, so treat this as coarse.

In [ ]:
!python scripts/reproduce.py --config configs/qwen3-1p7b-to-4b.yaml --task hellaswag --limit 250
!python scripts/reproduce.py --config configs/qwen3-1p7b-to-4b.yaml --task arc_challenge --limit 250

### Step 5: smoke test (hand read)
10 harmful, 5 plain benign, 5 borderline-benign (over-refusal check) chat prompts. Label each
completion refuse / comply / incoherent in `smoke.md`. Look for: refusals flipping to
compliance, coherence collapse, benign prompts newly refused. Outputs stay in `results/`
(git-ignored), since they may contain harmful text. Download them before the runtime ends.

In [ ]:
!python scripts/smoke.py --config configs/qwen3-1p7b-to-4b.yaml

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/smoke/qwen3-1p7b-to-4b/qwen3-1p7b-to-4b_k8/smoke.md', encoding='utf-8').read()))

## Decision: upgrade to Pro+ and an A100?

| What the L4 showed | Next |
|---|---|
| Step 1 or 3 fails, or mapped accuracy is about the 1.7B's | Debug here; an A100 will not help |
| Steps 1 to 4 plausible, smoke test shows nothing | Still worth the A100 anchor: a null result needs a validated pipeline |
| Steps 1 to 4 plausible, smoke test shows refusal changes | Upgrade: Ministral 3 3B -> 8B anchor, then the 200 + 200 pilot |

Log the numbers in `research-log.md` either way.